In [11]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

In [2]:
df = pd.read_excel("step3result copy 2.xlsx") 
df

,Author 1: Relevant Article? (Yes/No),Author 2: Relevant Article? (Yes/No),month_published,title,keywords,abstract,PMID,medline_authors,URL,Downloaded,Text,APA_Citation,category
0,No,No,2021 Jun 3,Extra-axial haemorrhage in a patient with Alpo...,"['anaesthesia', 'pregnancy']",Extra-axial haemorrhage following epidural ana...,34083183,"['Wijayanayaka S', 'Guha A', 'Sivanesan K', 'V...",https://europepmc.org/backend/ptpmcrender.fcgi...,True,Case report\nExtra- axial haemorrhage in a pat...,"Wijayanayaka, S., Guha, A., Sivanesan, K., & V...",'Neurological Disorders'
1,No,No,2015 Mar,"Connective tissue, Ehlers-Danlos syndrome(s), ...","['POTS', 'cervical spine', 'connective tissue'...",Ehlers-Danlos syndrome (EDS) is an umbrella te...,25655119,"['Castori M', 'Morlino S', 'Ghibellini G', 'Ce...",https://europepmc.org/backend/ptpmcrender.fcgi...,True,Cheema et al. The Journal of Headache\nThe Jou...,"Castori, M., Morlino, S., Ghibellini, G., Cell...",'Neurological Disorders'
2,No,No,2018 Jan 18,Case 2-2018. A 41-Year-Old Woman with Vision D...,"['Adult', 'Diagnosis, Differential', 'Female',...",No abstract available,29342391,"['Cestari DM', 'Cunnane ME', 'Rizzo JF 3rd', '...",https://libkey.io/libraries/731/articles/17711...,False,"282 n engl j med 378;3 nejm.org January 18, 20...","Cestari, D. M., Cunnane, M. E., Rizzo III, J. ...",No abstract available
3,No,No,2016 Jun,[Safe and Effective Analgesia with Bilateral C...,"['Anesthesia, Epidural', 'Aortic Aneurysm, Abd...",A patient with Marfan syndrome underwent emerg...,27483660,"['Katakura Y', 'Sakurai A', 'Endo M', 'Hamada ...",https://europepmc.org/backend/ptpmcrender.fcgi...,True,TYPE CaseReport\nPUBLISHED 13March2023\nDOI 10...,"Katakura, Y., Sakurai, A., Endo, M., Hamada, T...",'Other Conditions'
4,No,No,2014 Sep,Joint hypermobility and headache: the glue tha...,"['Ehlers-Danlos syndrome', ""Marfan's syndrome""...",BACKGROUND: Past studies have reported that co...,24958300,"['Martin VT', 'Neilson D']",https://europepmc.org/backend/ptpmcrender.fcgi...,True,1212081\nresearch-article2023 GPHXXX10.1177/23...,"Martin, V. T., & Neilson, D. (2014). Joint hyp...",'Neurological Disorders'
5,No,No,1999 Jan-Feb,Progressive systemic sclerosis: intrathecal pa...,"['Aged', 'Analgesics, Opioid/*administration &...","BACKGROUND AND OBJECTIVES: At present, there i...",9952101,"['Lundborg CN', 'Nitescu PV', 'Appelgren LK', ...",Not available,False,Text not available,"Lundborg, C. N., Nitescu, P. V., Appelgren, L....",'Autoimmune Disorders'
6,No,No,2015 May,Marfan syndrome presenting with postpartum aor...,"['Adult', 'Aortic Dissection/complications/*di...",No abstract available,25790894,"['Humphrey V', 'Falzon D', 'Clark V']",https://libkey.io/libraries/731/articles/53923...,False,References 1. Auerbach M. IV Iron in pregnancy...,"Humphrey, V., Falzon, D., & Clark, V. (2015). ...",No abstract available
7,No,No,1948 Sep-Oct,[Injection of the sympathetic superior cervica...,['*ANESTHESIA AND ANESTHETICS/spinal--complica...,No abstract available,18893366,"['MOYSON F', 'LAMBIOTTE C']",Not available,False,Text not available,"MOYSON, F., & LAMBIOTTE, C. (1948). Injection ...",No abstract available
8,No,No,2015 Dec,SLE Neuropathy-Anything New?,"['Guillain-Barre Syndrome', 'Headache', 'Human...",SLE (systemic lupus erythematosus) is a multis...,27666897,['Londhey VA'],https://europepmc.org/backend/ptpmcrender.fcgi...,True,REVIEW\npublished:04July2022\ndoi:10.3389/fimm...,"Londhey, V. A. (2015). SLE Neuropathy-Anything...",'Neurological Disorders'
9,No,No,2021 May,Long-Term Safety and Efficacy of Anifrolumab i...,"['Adult', 'Antibodies, Antinuclear/immunology'...",OBJECTIVE: To investigate long-term safety and...,33225631,"['Chatham WW', 'Furie R', 'Saxena A', 'Brohawn...",https://europepmc.org/backend/ptpmcrender.fcgi...,True,"Arthritis & Rheumatology\nVol. 73, No. 5, May ...","Chatham, W. W., Furie, R., Saxena, A., Brohawn...",'Autoimmune Disorders'


In [3]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [13]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
# from langchain_text_splitters import CharacterTextSplitter

CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT16k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-35-turbo-16",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Use APA-style in-text citations throughout the paragraph. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragraph summarizing the overall contents, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [23]:
# use abtract when text is not available.
df['Text'] = df.apply(lambda row: row['abstract'] if row['Text'] == 'Text not available' else row['Text'], axis=1)

output = []
for current_category in categories:
    filtered_rows = df[(df['category'] == current_category)]

    article_summaries = []
    for _, row in filtered_rows.iterrows():
        text_spilt = text_splitter.create_documents([row.Text])
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content="APA Citation: " + row.APA_Citation + "\n\nArticle text:\n\n" +text_spilt,
        ).to_messages())

        summary_to_keep = f"APA Citation: {row.APA_Citation}\n\n Summary: {result.content}\n\n --- "
        print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    
    print(result.content)
    output.append(result.content)
    print("************")
    

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: f06be44b********************493b. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [21]:
print(output)

['The scoping review of articles on neurological disorders found several relevant studies. Wijayanayaka et al. (2021) presented a case of an Alport syndrome patient who developed an extra-axial hemorrhage following epidural anesthesia, suggesting a potential link between connective tissue diseases and post-dural puncture headache. Castori et al. (2015) compared the phenotype of new daily persistent headache (NDPH) to transformed chronic daily headache (T-CDH) and found differences in symptoms and treatment response, indicating that NDPH may have distinct sub-phenotypes. Martin and Neilson (2014) explored the barriers physicians face in diagnosing and managing patients with Ehlers-Danlos Syndrome (EDS), highlighting the need for additional support and training. However, Londhey (2015) and Golfinos et al. (1995) did not directly address the question of connective tissue diseases contributing to post-dural spinal puncture headache. Overall, these studies suggest a potential association be

In [19]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

In [18]:
!pip install --quiet langchain_experimental langchain_openai




[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip


In [20]:
text_splitter = SemanticChunker(OpenAIEmbeddings())